In [ ]:
import itertools
from ast import literal_eval
import os

import numpy as np
import pandas as pd
from scipy.integrate import trapezoid

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from mg_sampling import PROJECT_DIR
from mg_sampling.constants import RT
from mg_sampling.tools import read_plumed, read_plumed_fes, range_to_edges, edges_to_range, finitemin, finitemax, block_analysis, read_wsv, demux, read_performance
from mg_sampling.bias import bias_fn

LABELS = {
    'amber_tip3p': "Amber/TIP3P",
    'mMg_tip3p': "microMg/TIP3P",
    'nMg_tip3p': "nanoMg/TIP3P", 
    'mMg_opc': "microMg/OPC",
    'ecc_spce': "AmberECC/SPCE",
}

FORCEFIELDS_ARCHIVE = ['amber_tip3p', 'mMg_tip3p', 'nMg_tip3p', 'mMg_opc', 'ecc_spce']
FORCEFIELDS = ['mMg_tip3p']

### Evolution of the number of inner-bound Mg²⁺ (Replex vs. PlainMD)

In [ ]:
n_inner_pdb = 14

system = '7ez0'
ff = 'mMg_tip3p'
replicate = 0
nrep = 8

fig, ax = plt.subplots(figsize=(5, 4))

# Replex
d_mg_op_dmx = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

d_mg_op = demux(d_mg_op_dmx, repindex_file=PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep00/bias_index.xvg')

time = d_mg_op[0, :, 0] / 1000  # ns
is_bound = (d_mg_op[0, :, 2:] < 0.28)

ax.plot(time, is_bound.sum(axis=-1), color='tomato', lw=1, label='μMg (REx)')

# Plain MD

ff = 'mMg_tip3p'
nrep = 8

d_mg_op = [
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
]
n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

time = d_mg_op[0, :, 0] / 1000  # ns
is_bound = (d_mg_op[:, :, 2:] < 0.28)


n_bound_avg = is_bound.sum(axis=-1).mean(axis=0)
n_bound_err = 1.96 * is_bound.sum(axis=-1).std(axis=0) / np.sqrt(nrep - 1)
ax.plot(time, n_bound_avg, color='mediumpurple', lw=1, label='μMg (MD)')
ax.fill_between(time, n_bound_avg - n_bound_err, n_bound_avg + n_bound_err, fc='mediumpurple', alpha=0.25)


ff = 'nMg_tip3p'
nrep = 8

d_mg_op = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

time = d_mg_op[0, :, 0] / 1000  # ns
is_bound = (d_mg_op[:, :, 2:] < 0.28)


n_bound_avg = is_bound.sum(axis=-1).mean(axis=0)
n_bound_err = 1.96 * is_bound.sum(axis=-1).std(axis=0) / np.sqrt(nrep - 1)
ax.plot(time, n_bound_avg, color='gray', lw=1, label='nMg (MD)')
ax.fill_between(time, n_bound_avg - n_bound_err, n_bound_avg + n_bound_err, fc='gray', alpha=0.25)


ax.axhline(n_inner_pdb, ls='dashed', lw=1, color='silver')
ax.text(500, n_inner_pdb - 1, 'PDB', style='italic', color='silver', ha='left', va='top')

ax.legend(frameon=False)
ax.set_xlim(-10, 600)
ax.set_ylim(0, 100)
ax.set_xlabel('Time [ns]')
ax.set_ylabel(r'Mg$^{\text{2+}}$ inner-bound to $\mathrm{O_P}$')

plt.savefig('figures/inner_mg_op_timeseries.pdf')

In [ ]:
def cumulative_unique(seq):
    """
    Cumulative number of unique elements along sequence.
    """
    out = []
    s = set()
    for x in seq:
        s.add(x)
        out.append(len(s))
    return out

n_inner_pdb = 14

system = '7ez0'
ff = 'mMg_tip3p'
replicate = 0
nrep = 8
cpu_per_rep = 8

fig, ax = plt.subplots(figsize=(5, 4))

# Replex
d_mg_op_dmx = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

d_mg_op = demux(d_mg_op_dmx, repindex_file=PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep00/bias_index.xvg')

is_bound = (d_mg_op[0, :, 2:] < 0.28)
n_unique = cumulative_unique(map(tuple, is_bound))

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single_bias/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h, n_unique, color='tomato', lw=1, label='μMg (REx)')

# Plain MD

ff = 'mMg_tip3p'
nrep = 8

d_mg_op = [
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
]
n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = np.sum([cumulative_unique(map(tuple, is_bound[i])) for i in range(nrep)], axis=0)

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h, n_unique, color='mediumpurple', lw=1, label='μMg (MD)')


ff = 'nMg_tip3p'
nrep = 8

d_mg_op = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = np.sum([cumulative_unique(map(tuple, is_bound[i])) for i in range(nrep)], axis=0)

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h, n_unique, color='gray', lw=1, label='nMg (MD)')

ax.legend(frameon=False)
ax.set_xlabel('CPU Hours [h]')
ax.set_ylabel(r'Unique Mg$^{2+}$ binding configuration')

plt.savefig('figures/unique_binding_timeseries.pdf')



In [ ]:
def cumulative_unique(seq):
    """
    Cumulative number of unique elements along sequence.
    """
    out = []
    s = set()
    for x in seq:
        s.add(x)
        out.append(len(s))
    return out

n_inner_pdb = 14

system = '7ez0'
ff = 'mMg_tip3p'
replicate = 0
nrep = 8
cpu_per_rep = 8

fig, ax = plt.subplots(figsize=(5, 4))

# Replex
d_mg_op_dmx = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

d_mg_op = demux(d_mg_op_dmx, repindex_file=PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep00/bias_index.xvg')

is_bound = (d_mg_op[:, :, 2:] < 0.28)
# n_unique = cumulative_unique(map(tuple, is_bound))
n_unique = np.sum([cumulative_unique(map(tuple, is_bound[i])) for i in range(nrep)], axis=0)

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single_bias/n1.log')  # hour/ns

ax.plot(time, n_unique, color='tomato', lw=1, label='μMg (REx)')

# Plain MD

ff = 'mMg_tip3p'
nrep = 8

d_mg_op = [
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
]
n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = np.sum([cumulative_unique(map(tuple, is_bound[i])) for i in range(nrep)], axis=0)

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns

ax.plot(time, n_unique, color='mediumpurple', lw=1, label='μMg (MD)')


ff = 'nMg_tip3p'
nrep = 8

d_mg_op = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = np.sum([cumulative_unique(map(tuple, is_bound[i])) for i in range(nrep)], axis=0)

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns

ax.plot(time, n_unique, color='gray', lw=1, label='nMg (MD)')

ax.legend(frameon=False)
ax.set_xlabel('Simulation time [ns]')
ax.set_ylabel(r'Unique Mg$^{2+}$ binding configuration')

plt.savefig('figures/unique_binding_timeseries.pdf')

In [ ]:
n_inner_pdb = 14

system = '7ez0'
ff = 'mMg_tip3p'
replicate = 0
nrep = 8
cpu_per_rep = 8

fig, ax = plt.subplots(figsize=(5, 4))

# Replex
d_mg_op_dmx = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

d_mg_op = demux(d_mg_op_dmx, repindex_file=PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep00/bias_index.xvg')

is_bound = (d_mg_op[:, :, 2:] < 0.28)
# n_unique = cumulative_unique(map(tuple, is_bound))
n_unique = np.sum([cumulative_unique(map(tuple, is_bound[i])) for i in range(nrep)], axis=0)

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single_bias/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h, n_unique, color='tomato', lw=1, label='μMg (REx)')

# Plain MD

ff = 'mMg_tip3p'
nrep = 8

d_mg_op = [
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
]
n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = np.sum([cumulative_unique(map(tuple, is_bound[i])) for i in range(nrep)], axis=0)

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h, n_unique, color='mediumpurple', lw=1, label='μMg (MD)')


ff = 'nMg_tip3p'
nrep = 8

d_mg_op = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = np.sum([cumulative_unique(map(tuple, is_bound[i])) for i in range(nrep)], axis=0)

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h, n_unique, color='gray', lw=1, label='nMg (MD)')

ax.legend(frameon=False)
ax.set_xlabel('CPU Hours [h]')
ax.set_ylabel(r'Unique Mg$^{2+}$ binding configuration')

plt.savefig('figures/unique_binding_timeseries.pdf')



Total entropy of the average binding vector.

In [ ]:
from scipy.stats import entropy

def cumulative_entropy(x):
    return [entropy(x[:i].mean(axis=0)) for i in range(1, len(x))]


n_inner_pdb = 14

system = '7ez0'
ff = 'mMg_tip3p'
replicate = 0
nrep = 8
cpu_per_rep = 8

fig, ax = plt.subplots(figsize=(5, 4))

# Replex
d_mg_op_dmx = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

d_mg_op = demux(d_mg_op_dmx, repindex_file=PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep00/bias_index.xvg')

is_bound = (d_mg_op[0, :, 2:] < 0.28)
# n_unique = cumulative_unique(map(tuple, is_bound))
n_unique = cumulative_entropy(is_bound)

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single_bias/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h[1:], n_unique, color='tomato', lw=1, label='μMg (REx)')

# Plain MD

ff = 'mMg_tip3p'
nrep = 8

d_mg_op = [
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
]
n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = cumulative_entropy(is_bound.mean(axis=0))

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h[1:], n_unique, color='mediumpurple', lw=1, label='μMg (MD)')


ff = 'nMg_tip3p'
nrep = 8

d_mg_op = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = cumulative_entropy(is_bound.mean(axis=0))

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h[1:], n_unique, color='gray', lw=1, label='nMg (MD)')

ax.legend(frameon=False)
ax.set_xlabel('CPU Hours [h]')
ax.set_ylabel(r'Binding distribution entropy')

plt.savefig('figures/unique_binding_timeseries.pdf')

Vendi score.

This is sensitive to number of samples so we need to compare at fixed sample count…
-> sample with replacement 

In [ ]:
from vendi_score import vendi
from scipy.spatial.distance import cosine, hamming, pdist, squareform

def vendi_score(x, metric='hamming', n_samples=50, n_bootstrap=5):
    res = []
    for _ in range(n_bootstrap):
        inds = np.random.choice(len(x), size=n_samples)
        _x = x[inds]
        res.append(vendi.score_K(1 - squareform(pdist(x, metric=metric))))
    avg = np.mean(res)
    stderr = np.std(res)
    return avg, stderr

n_inner_pdb = 14

system = '7ez0'
ff = 'mMg_tip3p'
replicate = 0
nrep = 8
cpu_per_rep = 8

stride = 100

fig, ax = plt.subplots(figsize=(5, 4))

# Replex
d_mg_op_dmx = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

d_mg_op = demux(d_mg_op_dmx, repindex_file=PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep00/bias_index.xvg')

is_bound = (d_mg_op[:, :, 2:] < 0.28)

y, yerr = np.array([vendi_score(is_bound[0, 0:t, :]) for t in range(1, is_bound.shape[1], stride)]).T

time = d_mg_op[0, ::stride, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single_bias/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.errorbar(cpu_h[:-1], y, yerr=yerr, color='tomato', lw=1, label='μMg (REx)')

# Plain MD

ff = 'mMg_tip3p'
nrep = 8

d_mg_op = [
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
]
n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])[:, ::stride]

is_bound = (d_mg_op[:, :, 2:] < 0.28)
y, yerr = np.array([vendi_score(is_bound[:, 0:t, :].reshape(-1, is_bound.shape[-1])) for t in range(1, is_bound.shape[1], stride)]).T

time = d_mg_op[0, ::stride, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.errorbar(cpu_h[:-1], y, yerr=yerr, color='mediumpurple', lw=1, label='μMg (MD)')


ff = 'nMg_tip3p'
nrep = 8

d_mg_op = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])[:, ::stride]

is_bound = (d_mg_op[:, :, 2:] < 0.28)
y, yerr = np.array([vendi_score(is_bound[:, 0:t, :].reshape(-1, is_bound.shape[-1])) for t in range(1, is_bound.shape[1], stride)]).T

time = d_mg_op[0, ::stride, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.errorbar(cpu_h[:-1], y, yerr=yerr, color='gray', lw=1, label='nMg (MD)')

ax.legend(frameon=False)
ax.set_xlabel('CPU Hours [h]')
ax.set_ylabel(r'Vendi score (Hamming)')

plt.savefig('figures/unique_binding_timeseries.pdf')

In [ ]:
is_bound.shape

In [ ]:
n_inner_pdb = 14

system = '7ez0'
ff = 'mMg_tip3p'
replicate = 0
nrep = 8
cpu_per_rep = 8

fig, ax = plt.subplots(figsize=(5, 4))

# Replex
d_mg_op_dmx = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

for stride in [2, 5, 10, 20, 50, 100, 200]: 
    d_mg_op = demux(d_mg_op_dmx, repindex_file=PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep00/bias_index.xvg')[:, ::stride]

    is_bound = (d_mg_op[:, :, 2:] < 0.28)

    n_unique = [vendi_score(is_bound[0, 0:t, :]) for t in range(1, is_bound.shape[1])]

    time = d_mg_op[0, :, 0] / 1000  # ns
    perf = read_performance(PROJECT_DIR / f'benchmark/single_bias/n1.log')  # hour/ns
    cpu_h = time * perf * nrep * cpu_per_rep

    ax.plot(cpu_h[:-1], n_unique, lw=1, label=f"{stride}")
ax.legend()

In [ ]:

n_inner_pdb = 14

system = '7ez0'
ff = 'mMg_tip3p'
replicate = 0
nrep = 8
cpu_per_rep = 8

stride = 100

fig, ax = plt.subplots(figsize=(5, 4))

# Replex
d_mg_op_dmx = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

d_mg_op = demux(d_mg_op_dmx, repindex_file=PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep00/bias_index.xvg')[:, ::stride]

is_bound = (d_mg_op[:, :, 2:] < 0.28)

n_unique = [vendi_score(is_bound[0, 0:t, :], metric='cosine') for t in range(1, is_bound.shape[1])]

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single_bias/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h[:-1], n_unique, color='tomato', lw=1, label='μMg (REx)')

# Plain MD

ff = 'mMg_tip3p'
nrep = 8

d_mg_op = [
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
]
n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])[:, ::stride]

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = [vendi_score(is_bound[:, 0:t, :].reshape(-1, is_bound.shape[-1]), metric='cosine') for t in range(1, is_bound.shape[1])]

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h[:-1], n_unique, color='mediumpurple', lw=1, label='μMg (MD)')


ff = 'nMg_tip3p'
nrep = 8

d_mg_op = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])[:, ::stride]

is_bound = (d_mg_op[:, :, 2:] < 0.28)
n_unique = [vendi_score(is_bound[:, 0:t, :].reshape(-1, is_bound.shape[-1]), metric='cosine') for t in range(1, is_bound.shape[1])]

time = d_mg_op[0, :, 0] / 1000  # ns
perf = read_performance(PROJECT_DIR / f'benchmark/single/n1.log')  # hour/ns
cpu_h = time * perf * nrep * cpu_per_rep

ax.plot(cpu_h[:-1], n_unique, color='gray', lw=1, label='nMg (MD)')

ax.legend(frameon=False)
ax.set_xlabel('CPU Hours [h]')
ax.set_ylabel(r'Vendi score (Cosine)')

plt.savefig('figures/unique_binding_timeseries.pdf')